# Mangrove NDVI (monthly): STL trend + breakpoints

1. Load the monthly max-NDVI CSV and put every point on a regular monthly grid
2. Deseasonalise with STL and plot raw NDVI + STL trend per point
   * Group the lines into shape classes (stable / loss / gain) based on what the NDVI does, and map them
3. Detect breakpoints on the STL trend:
   * NDVI drops or rises **≥ 0.2 within 12 months**, and
   * it **stays** at the new level (does not recover) for at least `persist_years`
4. Table + bar plot of the breakpoints

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

## Config

In [ ]:
CONFIG = {
    "input_csvs": {"path/to/your_ndvi.csv"   },
    "output_dir": "WEBTOOL_MDP/analysis/satellite/ews_output_monthly",

    "max_gap_months": 2,       # interior gaps up to this many months are linearly interpolated
    "min_completeness": 0.6,   # drop a point if less than this fraction of months has data
    "stl_trend_months": 13,    # STL trend smoother length (odd, > 12). STL's default (~23 months)
                               # smears a sudden drop over ~2 years, so it would never reach 0.25 in
                               # 12 months. Shorter = sharper steps but a bit more wiggle.

    # Breakpoint rules
    "bp_threshold": 0.25,       # minimum NDVI drop/rise ...
    "bp_window_months": 12,    # ... within this many months
    "persist_years": 2,        # the new level must hold this long (no recovery) -> change this to 3, 5, ...

    # Shape classes (clustering of the NDVI lines)
    "baseline_years": 3,       # each line is compared with its own mean of the first N years
    "end_years": 3,           # length of the end range (years) that is compared with the start range
    "dip_ndvi": 0.0,          # monthly trend below this = open water -> not stable    "loss_threshold": -0.3,          # net NDVI change (end - start) at or below this = loss
    "loss_threshold": -0.3,
    "gain_threshold": 0.3,           # net NDVI change at or above this = gain (after drift correction)
    "turnover_threshold": 1.4,       # back-and-forth movement above this = high turnover (check the scatter plot)
    "water_ndvi": 0.25,        # never mangrove AND mean NDVI below this = non-mangrove (water / mudflat)
    "mangrove_ndvi": 0.3,     # mean NDVI above this = mangrove (check your stable lines)
    }



os.makedirs(CONFIG["output_dir"], exist_ok=True)

traj_colors = {1: '#12301f', 2: '#9fbf6a', 3: '#e31a1c', 4: '#1f78b4', 5: '#ffd92f'}
traj_labels = {1: 'Stable interior', 2: 'Stable edge', 3: 'Monotonic loss',
               4: 'Monotonic gain', 5: 'Ambiguous/high turnover'}

## 1. Load and regularise to monthly

In [ ]:
def load_long_csv(path):
    # point_id must be the stable per-point ID from the GEE script (not system:index,
    # which changes per month after flatten())
    df = pd.read_csv(path).rename(columns={"ndvi_max": "ndvi"})
    df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
    if "traj" not in df.columns:
        df["traj"] = np.nan
    return df[["point_id", "date", "ndvi", "traj", "lon", "lat", "n_images_that_month"]]


def regularize_to_monthly(df, max_gap_months=2, min_completeness=0.6):
    # One column per point, one row per calendar month (missing months become NaN)
    monthly = df.pivot_table(index="date", columns="point_id", values="ndvi").resample("MS").mean()
    interpolated = monthly.interpolate(method="linear", limit=max_gap_months, limit_area="inside")

    completeness = interpolated.notna().mean()
    keep = completeness[completeness >= min_completeness].index
    dropped = completeness[completeness < min_completeness].index.tolist()

    # STL can't handle NaNs: fill the remaining (longer) gaps from the neighbouring months
    cleaned = interpolated[keep].ffill().bfill()
    return cleaned, dropped


df = pd.concat([load_long_csv(path).assign(source=name)
                for name, path in CONFIG["input_csvs"].items()], ignore_index=True)
df["point_id"] = df["source"] + "_" + df["point_id"].astype(str)   # unique IDs across both files
print(df.groupby("source")["point_id"].nunique())                    # points per file

monthly, dropped = regularize_to_monthly(df, CONFIG["max_gap_months"], CONFIG["min_completeness"])
point_traj = df.groupby("point_id")["traj"].first()

print(f"{monthly.shape[1]} points kept, {len(dropped)} dropped (too incomplete): {dropped}")
print(f"period: {monthly.index.min():%Y-%m} to {monthly.index.max():%Y-%m}")

## 2. STL trend (deseasonalised NDVI)

`robust=True` so single outlier months (clouds, haze) don't pull the trend.

In [ ]:
trends = {}
for point_id in monthly.columns:
    series = monthly[point_id].dropna()
    if len(series) < 24:   # STL needs at least 2 full yearly cycles
        continue
    trends[point_id] = STL(series, period=12, trend=CONFIG["stl_trend_months"],
                           robust=True).fit().trend.iloc[6:-12]   # cut STL edge effects (start 6, end 12 months)
print(f"STL trend computed for {len(trends)} points")

## 2b. Shape classes from the NDVI lines

The GEE classes don't always match what NDVI actually does, so here the points are grouped by the
**shape of their own STL trend**:

1. Each trend is averaged per year (less wiggle, same shape).
2. Each line is shifted so it starts at 0: minus its own mean of the first `baseline_years`.
   So a stable line at NDVI 0.7 and a stable line at 0.1 look the same; only the *change* counts.
3. **k-means** groups lines that look like each other (sizes don't have to be equal).
4. Each group gets a name from how much NDVI changed by the end: < −0.1 → loss, > +0.1 → gain, else stable.

First: all lines together.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for pid, trend in trends.items():
    ax.plot(trend.index, trend.values, color="tab:green", linewidth=0.8, alpha=0.3)
ax.set_title(f"STL trend of all {len(trends)} points")
ax.set_ylabel("NDVI (deseasonalised)")
ax.set_xlabel("Date")
ax.set_ylim(-1,1)
fig.savefig(os.path.join(CONFIG["output_dir"], f"STL_trends(n={len(trends)}).png"), dpi=120)
fig.tight_layout()
plt.show()

## classification cell below

In [ ]:
trend_df = pd.DataFrame(trends)                      # rows = months, columns = points
yearly = trend_df.resample("YS").mean()              # one value per year (less wiggle)
months_per_year = trend_df.resample("YS").count().iloc[:, 0]
yearly = yearly[months_per_year == 12]               # only complete years


def shape_features(yearly, monthly):
    start = yearly.iloc[:CONFIG["baseline_years"]].median()     # start range
    end = yearly.iloc[-CONFIG["end_years"]:].median()           # end range
    net_change = end - start
    return pd.DataFrame({
        "net_change": net_change,
        "turnover": yearly.diff().abs().sum() - net_change.abs(),   # back-and-forth movement
        "start_ndvi": start,
        "end_ndvi": end,
        "mean_ndvi": yearly.mean(),
        "min_ndvi": yearly.min(),                                   # lowest yearly value
        "min_monthly": monthly.min(),                               # lowest monthly trend value
        "years_lost": (yearly < CONFIG["mangrove_ndvi"]).apply(
            lambda col: col.groupby((~col).cumsum()).sum().max()),  # longest run of years below mangrove level
    })

def classify(row):
    m = CONFIG["mangrove_ndvi"]
    was_mangrove = row["start_ndvi"] >= m
    is_mangrove = row["end_ndvi"] >= m
    if not was_mangrove and not is_mangrove and row["mean_ndvi"] < CONFIG["water_ndvi"]:
        return "non-mangrove"            # water / mudflat, never mangrove
    if row["net_change"] <= CONFIG["loss_threshold"]:
        return "loss"
    if row["net_change"] >= CONFIG["gain_threshold"]:
        return "gain"
    if was_mangrove and is_mangrove and row["years_lost"] >= CONFIG["persist_years"]:
        return "loss + recovery"         # mangrove at start and end, lost for >= persist_years in between
    unstable = (row["turnover"] >= CONFIG["turnover_threshold"]      # wobbles a lot
                or row["min_ndvi"] < CONFIG["water_ndvi"]            # a whole year at water/mudflat level
                or row["min_monthly"] < CONFIG["dip_ndvi"])          # trend ever below 0 = open water
    if unstable:
        return "high turnover" if (was_mangrove or is_mangrove) else "non-mangrove"
    return "stable"


# --- Drift correction ---
# Pass 1 (uncorrected) is only used to find the stable mangrove points. Their median line should be
# flat; the slow rise it shows (more images per month in later years -> higher monthly max) is drift.
# We fit a straight line through it and subtract that from every point.
f0 = shape_features(yearly, trend_df)                  # pass 1: uncorrected
c0 = f0.apply(classify, axis=1)
ref_points = c0[(c0 == "stable") & (f0["mean_ndvi"] >= CONFIG["mangrove_ndvi"])].index
ref_curve = yearly[ref_points].median(axis=1)

t_years = yearly.index.year - yearly.index.year[0]
slope, intercept = np.polyfit(t_years, ref_curve.values, 1)
drift = pd.Series(slope * t_years, index=yearly.index)      # 0 at the start, grows linearly
print(f"drift from {len(ref_points)} stable mangrove points: {slope * 10:+.3f} NDVI per decade "
      f"({drift.iloc[-1]:+.2f} over the whole period)")

yearly_corr = yearly.sub(drift, axis=0)

# monthly drift-corrected trends, for the plots
t_months = (trend_df.index.year - yearly.index.year[0]) + (trend_df.index.month - 1) / 12
trend_corr = trend_df.sub(slope * t_months, axis=0)

# Pass 2: classify the drift-corrected lines
features = shape_features(yearly_corr, trend_corr)     # pass 2: drift-corrected
shape_class = features.apply(classify, axis=1).rename("shape_class")

shape_order = ["stable", "loss", "gain", "loss + recovery", "high turnover", "non-mangrove"]
shape_colors = {"stable": "darkgreen", "loss": "tab:red", "gain": "tab:blue", "loss + recovery": "tab:purple",
                "high turnover": "#ffd92f", "non-mangrove": "tab:gray"}

summary_shapes = features.groupby(shape_class)[["net_change", "turnover"]].mean().round(2)
summary_shapes.insert(0, "n_points", shape_class.value_counts())
summary_shapes = summary_shapes.reindex(shape_order)
summary_shapes["n_points"] = summary_shapes["n_points"].fillna(0).astype(int)
summary_shapes

In [ ]:
mean_stable = np.mean(shape_order=="stable")
print(mean_stable)

In [ ]:
lo, hi, u = CONFIG["loss_threshold"], CONFIG["gain_threshold"], CONFIG["turnover_threshold"]
fig, ax = plt.subplots(figsize=(7, 5))
for name in shape_order:
    m = shape_class == name
    ax.scatter(features.loc[m, "net_change"], features.loc[m, "turnover"], color=shape_colors[name],
               edgecolor="black", linewidth=0.3, s=35, label=f"{name} (n={m.sum()})")
ax.axvline(lo, color="black", linestyle="--", linewidth=0.8)
ax.axvline(hi, color="black", linestyle="--", linewidth=0.8)
ax.set_title(f"Shape classes of the NDVI lines (n={len(shape_class)})")
ax.plot([lo, hi], [u, u], color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("net NDVI change (end - start)")
ax.set_ylabel("turnover (back-and-forth NDVI change)")
ax.legend()
fig.savefig(os.path.join(CONFIG["output_dir"], f"Shape_classes_scatter_(n={len(shape_class)}).png"), dpi=120)
fig.tight_layout()
plt.show()

## Lines per class cell

In [ ]:
# Lines per shape class (thick black = class mean)
fig, axes = plt.subplots(len(shape_order), 1, figsize=(12, 3.5 * len(shape_order)), sharex=True, sharey=True)
for ax, name in zip(np.atleast_1d(axes), shape_order):
    pids = shape_class[shape_class == name].index
    for pid in pids:
        ax.plot(trend_corr.index, trend_corr[pid], color=shape_colors[name], linewidth=1, alpha=0.4)
    ax.plot(trend_corr.index, trend_corr[pids].mean(axis=1), color="black", linewidth=1.2, label="class mean")
    ax.set_title(f"{name} (n={len(pids)})")
    ax.set_ylabel("NDVI (deseasonalised)")
    ax.axhline(0, color="grey", linewidth=0.6, linestyle="--", label="NDVI=0")
    ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["output_dir"], f"shape_classes_lines(n={len(shape_class)}).png"), dpi=120)
plt.show()

In [ ]:
# Map of the points, coloured by shape class
coords = df.groupby("point_id")[["lon", "lat"]].first().loc[shape_class.index]

fig, ax = plt.subplots(figsize=(9, 8))
for name in shape_order:
    m = shape_class == name
    ax.scatter(coords.loc[m, "lon"], coords.loc[m, "lat"], color=shape_colors[name], s=45,
               edgecolor="black", linewidth=0.3, label=f"{name} (n={m.sum()})")
ax.set_aspect(1 / np.cos(np.deg2rad(coords["lat"].mean())))   # so the map isn't stretched
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Spatial map of shape classes (n={len(shape_class)})")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["output_dir"], f"shape_classes_map(n={len(shape_class)}).png"), dpi=150)

# Save points with coordinates and shape class
points_out = coords.join(shape_class).join(features.round(3))
points_out.to_csv(os.path.join(CONFIG["output_dir"], f"shape_classes_points(n={len(shape_class)}).csv"),
                  index_label="point_id")

plt.show()

In [ ]:
# How do the new shape classes compare with the GEE classes?
compare = pd.crosstab(point_traj.reindex(shape_class.index).map(traj_labels).rename("GEE class"),
                      shape_class)[shape_order]
pd.DataFrame({"point_id": shape_class.index, "traj": point_traj.reindex(shape_class.index).values,
              "shape_class": shape_class.values}).to_csv(
    os.path.join(CONFIG["output_dir"], "shape_classes.csv"), index=False)
compare

## 3. Breakpoint detection

**Step 1 – big change:** for every month *t*, the change over the next 12 months is
Δ = trend(t + 12) − trend(t). Where |Δ| ≥ 0.2 we have a change. Consecutive months that meet this
(same direction) are one change; we keep the 12-month window with the largest change.
The breakpoint date is the month with the steepest step inside that window.

**Step 2 – no recovery:** after the change, the trend must stay at least 0.2 away from the level
before the change (median of the 12 months before) for every month of the next `persist_years`. A dip that recovers is rejected.

**Recovery of a rejected dip:** the rise after a rejected drop (or the other way around) is compared with
the level *before* the dip, so a simple recovery is not counted as a new breakpoint.

Changes in the last `persist_years` of the record can't be checked yet and are skipped.

In [ ]:
def find_breakpoints(trend, threshold, window, persist_years):
    persist = 12 * persist_years

    delta = trend.shift(-window) - trend                       # change over the next 12 months
    sign = np.sign(delta).where(delta.abs() >= threshold, 0).fillna(0)
    run_id = (sign != sign.shift()).cumsum()                   # consecutive months, same direction

    breakpoints = []
    rejected = None   # last rejected change: (type, level before it, end of its window)
    for _, run in sign.groupby(run_id):
        if run.iloc[0] == 0:
            continue
        t0 = delta.loc[run.index].abs().idxmax()               # strongest 12-month window
        t1 = t0 + pd.DateOffset(months=window)
        kind = "rise" if delta.loc[t0] > 0 else "drop"

        level_before = trend.loc[t0 - pd.DateOffset(months=12):t0].median()   # 12 months before
        # NEW: only a change that comes soon after a rejected one (within persist_years) is its recovery
        if rejected is not None and rejected[0] != kind and t0 <= rejected[2] + pd.DateOffset(months=persist):
            level_before = rejected[1]

        # does the trend stay >= threshold away from that level for persist_years after the change?
        after = trend.loc[t1:t1 + pd.DateOffset(months=persist)]
        if len(after) <= persist:                              # not enough data left to check
            continue
        shift = (after - level_before) if kind == "rise" else (level_before - after)
        if (shift >= threshold).all():
            steps = trend.loc[t0:t1].diff()
            bp_date = steps.idxmax() if kind == "rise" else steps.idxmin()
            breakpoints.append({"type": kind, "breakpoint_date": bp_date,
                                "window_start": t0, "window_end": t1,
                                "magnitude": round(trend.loc[t1] - trend.loc[t0], 3)})
            rejected = None
        else:
            rejected = (kind, level_before, t1)                # NEW: also remember when it happened
    return breakpoints


records = []
for point_id, trend in trends.items():
    for bp in find_breakpoints(trend, CONFIG["bp_threshold"], CONFIG["bp_window_months"],
                               CONFIG["persist_years"]):
        records.append({"point_id": point_id, "traj": point_traj.get(point_id), **bp})

breakpoints_df = pd.DataFrame(records, columns=["point_id", "traj", "type", "breakpoint_date",
                                                "window_start", "window_end", "magnitude"])
breakpoints_df = breakpoints_df[breakpoints_df["point_id"].map(shape_class) != "non-mangrove"]

print(f"{len(breakpoints_df)} breakpoints in {breakpoints_df['point_id'].nunique()} "
      f"of {len(trends)} points")

## 4. Plot per point: raw NDVI + STL trend + breakpoints

Red = drop, blue = rise. The shaded band is the 12-month window of the change.

In [ ]:
for point_id, trend in trends.items():
    series = monthly[point_id]
    traj = point_traj.get(point_id)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(series.index, series.values, color="lightgray", linewidth=1, label="raw NDVI")
    ax.plot(trend.index, trend.values, color=shape_colors.get(shape_class.get(point_id), "tab:green"), linewidth=1.8, label="STL trend (deseasonalised)")

    for _, bp in breakpoints_df[breakpoints_df["point_id"] == point_id].iterrows():
        c = "tab:red" if bp["type"] == "drop" else "tab:blue"
        ax.axvspan(bp["window_start"], bp["window_end"], color=c, alpha=0.12)
        ax.axvline(bp["breakpoint_date"], color=c, linestyle="--",
                   label=f"{bp['type']} {bp['breakpoint_date']:%Y-%m} ({bp['magnitude']:+.2f})")

    ax.legend(fontsize=8, loc="lower left")
    ax.set_title(f"Point {point_id} - class: {shape_class.get(point_id, 'unclassified')}")
    ax.set_ylabel("NDVI")
    ax.set_ylim(-1, 1)
    fig.savefig(os.path.join(CONFIG["output_dir"], f"point_{point_id}.png"), dpi=120)
    # plt.show()

## 6. Breakpoint table

In [ ]:
table = breakpoints_df.sort_values("breakpoint_date").copy()
table["class"] = table["traj"].map(traj_labels)
table["breakpoint_year"] = table["breakpoint_date"].dt.year
table = table[["point_id", "traj", "class", "type", "breakpoint_date", "breakpoint_year", "magnitude"]]
table = table[table["type"] == "drop"]
table.to_csv(os.path.join(CONFIG["output_dir"], "breakpoints.csv"), index=False)
table.reset_index(drop=True)

## 7. Bar plot: breakpoints per year

In [ ]:
years = range(monthly.index.min().year, monthly.index.max().year + 1)
drops = table[table["type"] == "drop"]["breakpoint_year"].value_counts().reindex(years, fill_value=0)
rises = table[table["type"] == "rise"]["breakpoint_year"].value_counts().reindex(years, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(drops.index, drops.values, color="tab:red", label="drop")
# ax.bar(rises.index, rises.values, bottom=drops.values, color="tab:blue", label="rise")
ax.set_xlabel("Year")
ax.set_ylabel("Number of breakpoints")
ax.set_title(f"Breakpoints per year (≥{CONFIG['bp_threshold']} in {CONFIG['bp_window_months']} months, "
             f"lasting ≥{CONFIG['persist_years']} years)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["output_dir"], "breakpoints_per_year_complete.png"), dpi=120)

plt.show()